# Stage 2: BI Data Export and Tableau Integration

## 📚 Objectives
- Export processed RFM data and funnel metrics to CSV
- (Optional) Upload to AWS S3 for cloud storage
- Create Tableau dashboard for business visualization
- Implement AWS S3 client functions

## 📋 Agenda
1. Load processed data from Stage 1
2. Prepare dimensions and metrics for BI
3. Export to CSV files
4. (Optional) Upload to AWS S3
5. Connect to Tableau Desktop

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# TODO: Import AWS client from src if implementing cloud storage
# from src.data_pipeline.aws_client import AWSS3Client

In [ ]:
# TODO: Load RFM segments from Stage 1 output
# rfm_df = pd.read_csv('data/processed/rfm_segments.csv')
# print(f"Loaded {len(rfm_df)} customer records")

## 2. Prepare Data for BI (Fact and Dimension Tables)

In [ ]:
# TODO: Create dimension tables for Tableau
# - Customer Dimension: customer_id, segment, r_score, f_score, m_score
# - Date Dimension: date, month, quarter, year
# - Product Dimension: product_id, category, price

# TODO: Create fact tables for BI
# - Order Facts: order_id, customer_id, date, amount, etc.

## 3. Calculate Funnel Metrics

In [ ]:
# TODO: Calculate conversion funnel
# - Total Visitors
# - Add to Cart
# - Checkout
# - Payment Completed
# - Calculate step-to-step conversion rates

## 4. Export to CSV

In [ ]:
# TODO: Export dimension and fact tables to CSV
# rfm_df.to_csv('data/processed/rfm_for_tableau.csv', index=False)
# funnel_df.to_csv('data/processed/funnel_metrics.csv', index=False)

print("Data exported successfully!")

## 5. (Optional) Upload to AWS S3

## 5a. AWS Account Setup & S3 Configuration

### Step 1: Create AWS Account
1. Go to https://aws.amazon.com/
2. Click **"Create an AWS Account"** (top right)
3. Enter email address and create password
4. Choose **"Personal"** for account type
5. Enter billing information (free tier available for 12 months)
6. Verify phone number via SMS
7. Select **"Basic"** support plan (free)
8. Account activation takes ~5-15 minutes

### Step 2: Create S3 Bucket

**Via AWS Console (Easy):**
1. Log in to AWS Management Console (https://console.aws.amazon.com/)
2. Search for **"S3"** in the service search bar
3. Click **"Create bucket"** button
4. **Bucket name:** Enter unique name (e.g., `my-ecommerce-data-2025`)
   - Must be lowercase, alphanumeric + hyphens only
   - Must be globally unique across all AWS accounts
5. **Region:** Select closest to you (e.g., us-east-1)
6. **Block Public Access:** Keep all checked (secure by default)
7. Click **"Create bucket"**
8. Go to bucket → **Permissions tab** → **Bucket Policy** → Add policy to allow programmatic access

**Example bucket policy:**
```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": "arn:aws:iam::YOUR_ACCOUNT_ID:user/YOUR_USERNAME"
            },
            "Action": "s3:*",
            "Resource": [
                "arn:aws:s3:::YOUR_BUCKET_NAME",
                "arn:aws:s3:::YOUR_BUCKET_NAME/*"
            ]
        }
    ]
}
```

### Step 3: Create IAM User for Programmatic Access

1. Go to **IAM Service** (search in AWS console)
2. Click **"Users"** → **"Create user"**
3. **Username:** e.g., `ecommerce-app-user`
4. Check **"Provide user access to the AWS Management Console"** (optional)
5. Click **"Next"**
6. Select **"Attach policies directly"**
7. Search for **"AmazonS3FullAccess"** and check it
8. Click **"Create user"**
9. Click on user → **"Security credentials"** tab
10. **Create access key:**
    - Purpose: "Application running outside AWS"
    - Click **"Create access key"**
    - **IMPORTANT:** Save the Access Key ID and Secret Access Key (show only once!)

### Step 4: Connect to S3 from Python Code

**Method A: Using Environment Variables (Recommended - Most Secure)**

1. **On Windows:**
   - Press `Win + R`, type `SystemPropertiesAdvanced`, press Enter
   - Click **"Environment Variables"**
   - Under "User variables", click **"New"**
   - Variable name: `AWS_ACCESS_KEY_ID`
   - Variable value: (paste your Access Key ID)
   - Click OK, repeat for `AWS_SECRET_ACCESS_KEY`
   - **Restart your terminal/Python** for changes to take effect

2. **On Mac/Linux:**
   ```bash
   export AWS_ACCESS_KEY_ID=your_access_key_here
   export AWS_SECRET_ACCESS_KEY=your_secret_key_here
   ```
   Or add to `~/.bashrc` or `~/.zshrc` for persistence

**Method B: Using Credentials File**

1. Create file `~/.aws/credentials`:
   ```
   [default]
   aws_access_key_id = YOUR_ACCESS_KEY_ID
   aws_secret_access_key = YOUR_SECRET_ACCESS_KEY
   ```

2. Create file `~/.aws/config`:
   ```
   [default]
   region = us-east-1
   ```

**Method C: In Code (Less Secure - Only for Testing)**
```python
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id='YOUR_ACCESS_KEY',
    aws_secret_access_key='YOUR_SECRET_KEY',
    region_name='us-east-1'
)
```

### Step 5: Test Connection in Python

Run this cell to verify everything works:


In [ ]:
import boto3

# Test AWS connection
try:
    # Initialize S3 client (reads credentials from environment variables)
    s3 = boto3.client('s3')
    
    # List all buckets to verify connection
    response = s3.list_buckets()
    buckets = response['Buckets']
    
    print("✅ AWS Connection Successful!")
    print(f"Found {len(buckets)} bucket(s):")
    for bucket in buckets:
        print(f"  - {bucket['Name']}")
        
except Exception as e:
    print(f"❌ AWS Connection Failed: {str(e)}")
    print("Check your credentials are set correctly in environment variables")

In [ ]:
# TODO: Test AWS S3 upload functionality
# s3_client = AWSS3Client(bucket_name='my-bucket')
# s3_client.upload_dataframe_to_s3(rfm_df, 'data/rfm_segments.csv')
# s3_client.upload_dataframe_to_s3(funnel_df, 'data/funnel_metrics.csv')


### 📌 Implementation Task: Complete AWS S3 Client

**After testing S3 uploads in this notebook, implement in:** `src/data_pipeline/aws_client.py`

Complete these methods in the `AWSS3Client` class:

1. `__init__(bucket_name, aws_access_key_id=None, aws_secret_access_key=None)`
   - Initialize `boto3.client('s3')` with provided or environment credentials
   - Store bucket_name as instance variable

2. `upload_dataframe_to_s3(df, file_key)`
   - Convert DataFrame to CSV format (BytesIO stream)
   - Upload to S3 using `s3.put_object()`
   - Return True on success, False on failure
   - Example: `upload_dataframe_to_s3(rfm_df, 'data/rfm_segments.csv')`

3. `download_dataframe_from_s3(file_key)`
   - Download CSV from S3 using `s3.get_object()`
   - Parse with `pd.read_csv()`
   - Return DataFrame
   - Example: `rfm_df = s3_client.download_dataframe_from_s3('data/rfm_segments.csv')`

**Test each method here before moving to implementation file:**
- Create test credentials/environment variables
- Upload a sample DataFrame
- Download and verify structure matches

## 6. Tableau Integration Instructions

1. Open Tableau Desktop
2. Connect to data source: `data/processed/rfm_for_tableau.csv`
3. Create worksheets:
   - RFM Customer Segments (scatter plot: F vs M, colored by segment)
   - Customer Distribution by Segment (bar chart)
   - Conversion Funnel (funnel chart using funnel_metrics.csv)
4. Create dashboard combining all worksheets
5. Save as `dashboards/growth_dashboard.twb`

## 6a. Tableau Setup & Data Connection Guide

### Part 1: Download & Install Tableau Public (Free Version)

**Option A: Tableau Public (Cloud-based, Free Forever)**
1. Go to https://public.tableau.com/
2. Click **"Free Sign Up"** (top right)
3. Create account with email and password
4. Verify email
5. Download **"Tableau Public"** (desktop app)
   - Available for Windows and Mac
   - ~500MB download

**Option B: Tableau Desktop (Professional, 14-day Trial)**
1. Go to https://www.tableau.com/products/desktop/download
2. Enter email and choose platform
3. Download installer
4. Install and open - 14-day trial starts automatically
5. Great for testing before purchasing

---

## Part 2: Connecting Data to Tableau (Two Methods)

### **Method 1: Download CSV from S3, Then Connect (All Tableau Versions)**

**✅ Best for:** Tableau Public (free), beginners, simple setup
**⏱️ Time:** ~2-3 minutes

**Step 1: Download CSV from S3 in Python**
```python
from src.data_pipeline.aws_client import AWSS3Client

# Initialize S3 client
s3_client = AWSS3Client(bucket_name='your-bucket-name')

# Download CSV files to local directory
rfm_df = s3_client.download_dataframe_from_s3('data/rfm_segments.csv')
rfm_df.to_csv('data/processed/rfm_for_tableau.csv', index=False)

funnel_df = s3_client.download_dataframe_from_s3('data/funnel_metrics.csv')
funnel_df.to_csv('data/processed/funnel_for_tableau.csv', index=False)

print("✅ CSV files downloaded to data/processed/")
```

**Step 2: Connect CSV to Tableau**
1. Open Tableau
2. Click **"Connect to Data"** (left panel)
3. Select **"Text File"**
4. Browse to `data/processed/rfm_for_tableau.csv`
5. Click **"Open"**
6. Data preview appears - ready to visualize!

**Advantages:**
- ✅ Works with Tableau Public (free version)
- ✅ Fastest setup
- ✅ Full control over data before visualization

---

### **Method 2: Direct S3 Connection in Tableau Desktop (Advanced)**

**✅ Best for:** Tableau Desktop (paid/trial), direct cloud connection
**⏱️ Time:** ~5 minutes + AWS setup

**⚠️ Limitations:**
- ❌ Tableau Public (free) does NOT support S3 connectors
- ✅ Tableau Desktop (trial/paid) supports this
- ✅ Tableau Server/Online supports this

**Step 1: Set Up AWS Credentials (if not already done)**

Make sure you have:
- AWS Access Key ID
- AWS Secret Access Key
- S3 bucket name
- Set as environment variables (recommended) or saved in `~/.aws/credentials`

**Step 2: Connect in Tableau Desktop**

1. Open **Tableau Desktop**
2. Click **"Connect to Data"** (left panel)
3. Search for **"Amazon S3"** in the connector list
4. Click on it (may need to download connector first)
5. Enter credentials:
   - **AWS Access Key ID:** (your key)
   - **AWS Secret Access Key:** (your secret)
   - **Region:** (e.g., us-east-1)
6. Click **"Sign In"**
7. Select your S3 bucket
8. Browse to your file: `data/rfm_segments.csv`
9. Click **"OK"** to load data
10. Data loads directly from S3 - ready to visualize!

**Advantages:**
- ✅ Direct connection to S3 (no manual download)
- ✅ Live data updates if files change
- ✅ More scalable for large datasets
- ✅ Professional workflow

---

## Part 3: Data Source Connection Summary

| Scenario | Method | Tableau Version | Steps |
|----------|--------|---|---|
| **Learning, keeping it simple** | Download CSV → Connect to file | Public + Desktop | 2 |
| **Testing S3 upload worked** | Both methods | Desktop | 3 |
| **Production dashboard** | Direct S3 | Desktop/Server | 5 |
| **Sharing public dashboard** | Download CSV + Tableau Public | Public | 2 |

**Recommendation:** Start with **Method 1** (CSV download), then explore **Method 2** (direct S3) if you have Tableau Desktop trial.

---

### Part 4: Create Visualizations (Works with Both Methods)

Once data is connected (either method):

**Example 1: RFM Customer Segments Scatter Plot**

1. **Create new worksheet**
   - Click **"+"** tab → **"Sheet"**

2. **Drag fields:**
   - **Columns:** Drag `Frequency` (becomes x-axis)
   - **Rows:** Drag `Monetary` (becomes y-axis)
   - **Color:** Drag `Segment` (colors each point by segment)
   - **Size:** Drag `Customer_ID` count (bubble size by count)

3. **Result:** Scatter plot showing:
   - X-axis = Purchase frequency
   - Y-axis = Total spend
   - Color = Customer segment
   - Bubble size = Number of customers

**Example 2: Segment Distribution Bar Chart**

1. Create new sheet
2. **Drag fields:**
   - **Columns:** `Segment`
   - **Rows:** `Count of Customer_ID` (Tableau auto-counts)
   - **Color:** `Segment`

3. Result: Bar chart showing customer count per segment

**Example 3: Conversion Funnel**

1. Create new sheet
2. Load `funnel_metrics.csv` as new data source
3. **Drag fields:**
   - **Rows:** `Funnel_Step` (Visit → Cart → Checkout → Payment)
   - **Columns:** `User_Count`
   - Create bar chart with steps in order

### Part 5: Essential Tableau Concepts

| Concept | Meaning | Example |
|---------|---------|---------|
| **Dimension** | Categorical data (qualitative) | Segment, Region, Category |
| **Measure** | Numerical data (quantitative) | Revenue, Count, Average |
| **Rows/Columns** | Chart axes placement | Drag Segment to Rows, Revenue to Columns |
| **Marks** | Color, size, shape encoding | Color by Segment, Size by Revenue |
| **Filter** | Show only specific data | Show only "Champions" segment |
| **Aggregation** | How to sum data | SUM, AVG, COUNT, MIN, MAX |

### Part 6: Learning Resources

| Resource | Best For | Link |
|----------|----------|------|
| **Official Tableau Training** | Interactive videos & projects | https://www.tableau.com/learn/training |
| **Tableau YouTube Channel** | Quick tutorials (5-20 min) | https://www.youtube.com/user/tableaudesktop |
| **Tableau Public Gallery** | Real examples to explore | https://public.tableau.com/gallery |
| **Tableau Desktop Help** | Problem-solving & reference | Help menu → Tableau Desktop Help |
| **Tableau Community Forum** | Q&A with experts | https://community.tableau.com/ |

**Recommended Learning Path:**
1. **Start:** Watch "Getting Started with Tableau Desktop" (15 min)
2. **Practice:** Create scatter plot from your RFM data
3. **Explore:** Check Tableau Public Gallery for similar visualizations
4. **Deep Dive:** Official training on advanced topics (filters, parameters)

### Part 7: Publishing Dashboards

**With Tableau Public (Free):**
1. Create your worksheets and dashboard
2. File → **"Save as"**
3. Sign in with Tableau Public credentials
4. Dashboard becomes public and shareable via URL
5. Others can view, download, or interact

**With Tableau Desktop:**
1. File → **"Save As"** to publish to Tableau Server/Tableau Online
2. Or export as PDF/Image for static sharing



### Part 8: Create & Share Professional Dashboard (Company-Style)

#### **Overview: Dashboard Workflow in Companies**

In real companies:
1. **Data Analyst** creates visualizations in Tableau
2. **Combines them into a Dashboard** for overview
3. **Publishes to cloud** (Tableau Online/Server or Public)
4. **Shares link with stakeholders** (executives, managers, team)
5. **Others access via web browser** - no software needed!
6. **Auto-refreshes** with latest data

**You'll be doing the same thing with your RFM & Funnel data!**

---

#### **Step 1: Create Individual Worksheets**

First, create 3 separate visualizations:

**Worksheet 1: RFM Segments Overview**
- Name: "RFM Customer Segments"
- Type: Scatter plot (Frequency vs Monetary, colored by Segment)

**Worksheet 2: Segment Distribution**
- Name: "Customer Count by Segment"
- Type: Bar chart (Segment vs Count)

**Worksheet 3: Conversion Funnel**
- Name: "Conversion Funnel"
- Type: Bar chart (Funnel steps vs User count)

---

#### **Step 2: Combine Into a Dashboard**

1. Click **"+"** tab (next to sheet tabs)
2. Select **"Dashboard"**
3. **Drag worksheets onto dashboard canvas:**
   - Drag "RFM Customer Segments" → Top half (large)
   - Drag "Customer Count by Segment" → Bottom left
   - Drag "Conversion Funnel" → Bottom right

4. **Add interactivity (filters):**
   - Click "RFM Customer Segments" on dashboard
   - Click **"Filter"** icon
   - Select **"Segment"** field
   - Now users can filter by segment and see all charts update!

5. **Add title and description:**
   - Click **"Insert"** → **"Text"**
   - Add title: "E-Commerce Growth Analytics Dashboard"
   - Add subtitle: "Real-time RFM analysis and funnel metrics"

**Result:** Professional dashboard with 3 linked visualizations

---

#### **Step 3: Publish Dashboard (Create Shareable Link)**

**For Tableau Public (Free Cloud Version):**

1. File → **"Save As"** (or Ctrl+Shift+S)
2. In dialog, sign in with Tableau Public account
3. **Workbook name:** e.g., "E-Commerce RFM Analysis"
4. **Visibility:** Choose:
   - **Public** (anyone with link can view)
   - **Private** (only you can view)
5. Click **"Save"**
6. Tableau uploads and creates shareable link
7. Copy the URL, e.g.:
   ```
   https://public.tableau.com/app/profile/yourname/viz/E-CommerceRFMAnalysis/Dashboard
   ```

**For Tableau Desktop (Trial/Paid):**
1. Publish to **Tableau Server** or **Tableau Online**
2. File → **"Publish as..."**
3. Choose server and location
4. Get shareable link from server
5. Others access via browser

---

#### **Step 4: Share With Stakeholders (Company Use Case)**

**After publishing, share the link via:**
- 📧 Email: "Check out our RFM dashboard: https://public.tableau.com/..."
- 💬 Slack/Teams: Post dashboard link in channel
- 📊 Reports: Embed link in PowerPoint or docs
- 📱 Mobile: Others view on phone/tablet (responsive)

**Who sees it:**
- ✅ Anyone with the link (if Public)
- ✅ Team members (if Private on Tableau Online)
- ✅ Executives for decision-making
- ✅ Marketing for customer segment targeting

---

#### **Step 5: Dashboard Interactivity for End Users**

Users viewing your published dashboard can:

| Action | Result | Use Case |
|--------|--------|----------|
| **Click filter** | All charts update instantly | Manager filters to see "Champions" only |
| **Hover over data** | Tooltip shows details | Analyst sees exact numbers |
| **Click on bar/point** | Cross-filters other sheets | Sales clicks "At-Risk" segment, sees which products they bought |
| **Download as image** | Save visualization | Presenter includes in meeting slides |
| **Download as PDF** | Full dashboard snapshot | Email to executives |

**Example Company Workflow:**
```
CEO asks: "How many at-risk customers do we have?"
→ Open shared dashboard link
→ Click filter: Segment = "At-Risk"
→ See: 2,340 customers, avg $45 spend, purchased 1.2x this year
→ Download as PDF
→ Include in board meeting
```

---

#### **Step 6: Real Company Examples**

**What companies use this for:**

| Company | Use Case | Link Type |
|---------|----------|-----------|
| **E-Commerce** (like Olist) | RFM segmentation for targeting | Internal Tableau Server |
| **SaaS** | Customer retention by cohort | Tableau Online (shared) |
| **Retail** | Sales by region & product | Tableau Server (auto-refresh daily) |
| **Marketing** | Campaign performance metrics | Tableau Public (external partners) |
| **Finance** | Revenue forecasting | Tableau Server (secured) |

---

#### **What Your Dashboard Will Show**

**URL:** Something like:
```
https://public.tableau.com/app/profile/yourname/viz/ECommerceRFMAnalysis/Dashboard
```

**Viewers see:**
1. **Top Chart:** Scatter plot of all customers (dots colored by segment)
2. **Bottom Left:** Bar chart showing how many customers in each segment
3. **Bottom Right:** Funnel showing visitor → purchase conversion
4. **Filters:** Dropdown to select only specific segments
5. **Legends:** Color coding for Champions/Loyal/At-Risk/etc.

**They can:**
- ✅ Change filters to explore data
- ✅ Hover for detailed information
- ✅ Download as image/PDF
- ✅ Share the link with colleagues
- ✅ View on mobile/tablet

---

#### **Step 7: Dashboard Maintenance & Updates**

**In a company:**

| Activity | Frequency | Who Does It |
|----------|-----------|-------------|
| **Update source data** | Daily/Weekly | Data Engineer (runs Python scripts) |
| **Dashboard auto-refreshes** | Auto | Tableau Server (scheduled) |
| **Add new metrics** | Monthly | Data Analyst (updates sheets) |
| **Share with new team** | As needed | Dashboard owner |

**For your project:**
1. Studio runs `01_data_cleaning_and_rfm.ipynb` → generates CSV
2. CSV uploaded to S3
3. Tableau dashboard loads latest CSV
4. Everyone sees current data!

---

#### **Next Steps**

1. ✅ Create 3 worksheets (segments, count, funnel)
2. ✅ Combine into dashboard
3. ✅ Add filters
4. ✅ Publish to Tableau Public
5. ✅ Copy shareable URL
6. ✅ Share with friends/team
7. ✅ Monitor interactions
8. ✅ Update when data changes

**Your dashboard is now live, shareable, and interactive - just like in a real company!** 🎉